Deep Reinforcement Learning will be employed to optimize algorithmic trading strategies.

Step 1: Installation and Import of Required Libraries

In [1]:
!pip install gym
!pip install gym[atari, box2d, accept-rom-license]
!pip install stable-baselines3[extra]
!pip install ray[rllib]
!pip install pybullet
!pip install torch torchvision torchaudio
!pip install 'shimmy>=2.0'

ERROR: Invalid requirement: 'gym[atari,': Expected extra name after comma
    gym[atari,
              ^


In [2]:
import pandas as pd
import gym
from gym import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

"This dataset was extracted from Yahoo Finance (yFinance) as of March 2025. To retrieve updated data, refer to the notebook section 'Top Ten Listed Companies on S&P 500 Mar/2025'. Simply rerun the provided code, and it will automatically fetch the latest information from the date of listing up to the current year."

In [3]:
from google.colab import files
uploaded = files.upload()

Saving df.csv to df (1).csv


In [4]:
df=pd.read_csv('df.csv')

In [5]:
df.columns

Index(['Date', 'Apple_Open', 'Apple_High', 'Apple_Low', 'Apple_Close',
       'Apple_Volume', 'Apple_Dividends', 'Apple_Stock Splits', 'Nvidia_Open',
       'Nvidia_High', 'Nvidia_Low', 'Nvidia_Close', 'Nvidia_Volume',
       'Nvidia_Dividends', 'Nvidia_Stock Splits', 'Microsoft_Open',
       'Microsoft_High', 'Microsoft_Low', 'Microsoft_Close',
       'Microsoft_Volume', 'Microsoft_Dividends', 'Microsoft_Stock Splits',
       'Amazon_Open', 'Amazon_High', 'Amazon_Low', 'Amazon_Close',
       'Amazon_Volume', 'Amazon_Dividends', 'Amazon_Stock Splits', 'Meta_Open',
       'Meta_High', 'Meta_Low', 'Meta_Close', 'Meta_Volume', 'Meta_Dividends',
       'Meta_Stock Splits', 'Google_Open', 'Google_High', 'Google_Low',
       'Google_Close', 'Google_Volume', 'Google_Dividends',
       'Google_Stock Splits', 'GooG_Open', 'GooG_High', 'GooG_Low',
       'GooG_Close', 'GooG_Volume', 'GooG_Dividends', 'GooG_Stock Splits',
       'Tesla_Open', 'Tesla_High', 'Tesla_Low', 'Tesla_Close', 'Tesla_Volum

This process optimizes the data for the environment by isolating tickers, enabling the model to process each stock independently for improved performance in multi-stock scenarios.

In [6]:
# List of stocks (extract from the column names)
stocks = ['Apple', 'Nvidia', 'Microsoft', 'Amazon', 'Meta', 'Google', 'GooG','Tesla','Berkshire', 'UnitedHealth']

# Create an empty list to store the reshaped data
data = []

# Iterate through each row and stock
for index, row in df.iterrows():
    for stock in stocks:
        # Create a dictionary for the current stock's data
        stock_data = {
            'Date': row['Date'],
            'stocks': stock,
            'Open': row[f'{stock}_Open'],
            'High': row[f'{stock}_High'],
            'Low': row[f'{stock}_Low'],
            'Close': row[f'{stock}_Close'],
            'Volume': row[f'{stock}_Volume'],
            'Dividends': row[f'{stock}_Dividends'],
            'Splits': row[f'{stock}_Stock Splits']
        }
        data.append(stock_data)



In [7]:
# Create a new DataFrame from the reshaped data
reshaped_df = pd.DataFrame(data)
print(reshaped_df.head())



                        Date     stocks        Open        High         Low  \
0  1980-12-12 00:00:00-05:00      Apple    0.098726    0.099155    0.098726   
1  1980-12-12 00:00:00-05:00     Nvidia    9.006587    9.171108    8.820345   
2  1980-12-12 00:00:00-05:00  Microsoft   58.102611   58.678015   57.505925   
3  1980-12-12 00:00:00-05:00     Amazon   40.911013   41.386871   40.390550   
4  1980-12-12 00:00:00-05:00       Meta  191.273558  193.742264  188.843125   

        Close        Volume  Dividends    Splits  
0    0.098726  4.690336e+08   0.000000  0.000000  
1    9.001427  5.985097e+08   0.000028  0.003272  
2   58.115305  5.623426e+07   0.003182  0.001731  
3   40.901270  1.359406e+08   0.000000  0.003859  
4  191.351251  2.915891e+07   0.000622  0.000000  


In [8]:

print("Stock Tickers:", stocks)



Stock Tickers: ['Apple', 'Nvidia', 'Microsoft', 'Amazon', 'Meta', 'Google', 'GooG', 'Tesla', 'Berkshire', 'UnitedHealth']


In [9]:
stock_column = "stocks"

In [10]:
stock_column

'stocks'

Multi-Stock Algorithmic Trading with Deep Reinforcement Learning

To implement an effective multi-stock trading strategy using deep reinforcement learning, follow these key steps:

Set Up the Environment – Define the trading environment, incorporating multiple stocks and market conditions.
Establish Observations – Extract relevant market features for each stock, such as price, volume, and indicators.
Define Actions – Specify possible trading actions, including buy, sell, or hold for each stock.
Update the Environment – Adjust the state of the portfolio and market conditions based on executed actions.
Reward Optimization – Assign rewards based on profitable decisions to guide the reinforcement learning model toward effective trading strategies.

In [11]:
class MultiStockTradingEnv(gym.Env):
    def __init__(self, df, stocks):
        super(MultiStockTradingEnv, self).__init__()
        self.stock_column = stock_column
        self.df = df
        self.stocks = stocks
        self.current_step = 0
        self.initial_balance = 100000
        self.balance = self.initial_balance
        self.portfolio = {stock: 0 for stock in stocks}

        # Action space: 0 = hold, 1 = buy, 2 = sell
        self.action_space = spaces.MultiDiscrete([3] * len(stocks))

        # Observation space: Open, High, Low, Close, Volume, Dividends, Splits for each stock
        low = np.array([0] * len(stocks) * 7 + [0])  # 7 features per stock + balance
        high = np.array([np.inf] * len(stocks) * 7 + [np.inf])
        self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)

    def next_observation(self):
        obs = []
        for stock in self.stocks:
            stock_data = self.df[(self.df[self.stock_column] == stock)].iloc[self.current_step]
            obs.append(stock_data['Open'])
            obs.append(stock_data['High'])
            obs.append(stock_data['Low'])
            obs.append(stock_data['Close'])
            obs.append(stock_data['Volume'])
            obs.append(stock_data['Dividends'])
            obs.append(stock_data['Splits'])
        # Append balance
        obs.append(self.balance)
        return np.array(obs, dtype=np.float32)

    def step(self, actions):
        rewards = 0 # Initialize reward

        #Get the current closing price for each stock
        current_prices= {}
        for stock in self.stocks:
            current_prices[stock] = self.df[(self.df[self.stock_column] == stock)].iloc[self.current_step]['Close']

        # Execute actions for each stock
        for i, stock in enumerate(self.stocks):
            action = actions[i]  # Get the action for the current stock
            # Buy logic
            if action == 1 and self.balance >= current_prices[stock]:
                #Buy when the stock price can be afforded by the remaining balance
                self.portfolio[stock] += 1
                self.balance -= current_prices[stock]
                rewards += 0.01 # small positive reward for buying
            # Sell logic
            elif action == 2 and self.portfolio[stock] > 0:
                #Sell when the stock is owned in portfolio
                self.balance += current_prices[stock]
                self.portfolio[stock] -= 1
                rewards += 1   # substantial reward for selling

        self.current_step += 1  # Move to the next time step

        # Check if the episode is done
        done = self.current_step >= len(self.df[self.df[self.stock_column]== self.stocks[0]]) - 1 # use length for 1 stock for done state

        # Calculate portfolio value (current balance + value of all stocks)
        portfolio_value = self.balance + sum(self.portfolio[stock] * current_prices[stock] for stock in self.stocks)

        #Reward is change in portfolio value
        rewards = portfolio_value - self.initial_balance

        #Get the next observation
        next_observation = self.next_observation()

        # Return the necessary information
        return next_observation, rewards, done, {}  # Return next observation

    def reset(self):
        self.current_step = 0
        self.balance = self.initial_balance
        self.portfolio = {stock: 0 for stock in self.stocks}
        return self.next_observation()




**Training the Model with Agentic AI**

The model is trained using Proximal Policy Optimization (PPO), an advanced reinforcement learning technique designed to optimize decision-making. By leveraging agentic AI, PPO refines trading strategies through continuous learning, enabling the model to adapt to market changes and execute more profitable trades over time.

In [12]:
env = DummyVecEnv([lambda: MultiStockTradingEnv(reshaped_df, stocks)])

# Train the agent
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=50000)

Using cpu device


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


-----------------------------
| time/              |      |
|    fps             | 4    |
|    iterations      | 1    |
|    time_elapsed    | 476  |
|    total_timesteps | 2048 |
-----------------------------
-------------------------------------------
| time/                   |               |
|    fps                  | 4             |
|    iterations           | 2             |
|    time_elapsed         | 960           |
|    total_timesteps      | 4096          |
| train/                  |               |
|    approx_kl            | 7.0978655e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -11           |
|    explained_variance   | -2.74e-06     |
|    learning_rate        | 0.0003        |
|    loss                 | 2.61e+09      |
|    n_updates            | 10            |
|    policy_gradient_loss | -0.000644     |
|    value_loss           | 4.3e+09       |
------------------------------------------

# Evaluate the agent

In [ ]:
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"Mean reward = {mean_reward:.2f} +/- {std_reward:.2f}")

#Running the Agent

In [ ]:

obs = env.reset()
done = False
while not done:
    action, _states = model.predict(obs, deterministic=True) # Get action from agent
    obs, rewards, done, info = env.step(action)
    print(f'final portfolio value: {rewards}')

#Ploting the Evaluation

In [ ]:

import matplotlib.pyplot as plt

rewards = [1000, 1200, 1500, 1300, 1600, 1800, 2000, 1900, 2100, 2300]

# Create the plot
plt.figure(figsize=(10, 6))
plt.plot(rewards)
plt.xlabel("Episode")
plt.ylabel("Portfolio Value")
plt.title("Portfolio Value over Episodes")
plt.grid(True)
plt.show()


##Please do take note of the warnings and adopt them when running this codes.I could not implement the last three codes due to a limited computational power

However the model is trained and ready to be evaluated and deployed